This notebook is just to understand the raw data, and use that to decide preprocessing rules

In [23]:
# imports needed
import json
import pandas as pd
import re
from collections import Counter

In [7]:
# since dataset is huge, only doing eda for first 10000 lines
file_path = "../data/raw/arxiv-metadata-oai-snapshot.json"
data = []

with open(file_path, "r") as f:
    for i, line in enumerate(f):
        if i >= 10000:
            break
        data.append(json.loads(line))
arxiv_df = pd.DataFrame(data)


Just basic checks below like shape, column names, null value count, etc

In [8]:
arxiv_df.shape

(10000, 14)

In [9]:
arxiv_df.columns.tolist()

['id',
 'submitter',
 'authors',
 'title',
 'comments',
 'journal-ref',
 'doi',
 'report-no',
 'categories',
 'license',
 'abstract',
 'versions',
 'update_date',
 'authors_parsed']

In [10]:
arxiv_df.head()

,id,submitter,authors,title,comments,journal-ref,doi,report-no,categories,license,abstract,versions,update_date,authors_parsed
0,0704.0001,Pavel Nadolsky,"C. Bal\'azs, E. L. Berger, P. M. Nadolsky, C.-...",Calculation of prompt diphoton production cros...,"37 pages, 15 figures; published version","Phys.Rev.D76:013009,2007",10.1103/PhysRevD.76.013009,ANL-HEP-PR-07-12,hep-ph,None,A fully differential calculation in perturba...,"[{'version': 'v1', 'created': 'Mon, 2 Apr 2007...",2008-11-26,"[[Balázs, C., ], [Berger, E. L., ], [Nadolsky,..."
1,0704.0002,Louis Theran,Ileana Streinu and Louis Theran,Sparsity-certifying Graph Decompositions,To appear in Graphs and Combinatorics,None,None,None,math.CO cs.CG,http://arxiv.org/licenses/nonexclusive-distrib...,"We describe a new algorithm, the $(k,\ell)$-...","[{'version': 'v1', 'created': 'Sat, 31 Mar 200...",2008-12-13,"[[Streinu, Ileana, ], [Theran, Louis, ]]"
2,0704.0003,Hongjun Pan,Hongjun Pan,The evolution of the Earth-Moon system based o...,"23 pages, 3 figures",None,None,None,physics.gen-ph,None,The evolution of Earth-Moon system is descri...,"[{'version': 'v1', 'created': 'Sun, 1 Apr 2007...",2008-01-13,"[[Pan, Hongjun, ]]"
3,0704.0004,David Callan,David Callan,A determinant of Stirling cycle numbers counts...,11 pages,None,None,None,math.CO,None,We show that a determinant of Stirling cycle...,"[{'version': 'v1', 'created': 'Sat, 31 Mar 200...",2007-05-23,"[[Callan, David, ]]"
4,0704.0005,Alberto Torchinsky,Wael Abu-Shammala and Alberto Torchinsky,From dyadic $\Lambda_{\alpha}$ to $\Lambda_{\a...,None,"Illinois J. Math. 52 (2008) no.2, 681-689",None,None,math.CA math.FA,None,In this paper we show how to compute the $\L...,"[{'version': 'v1', 'created': 'Mon, 2 Apr 2007...",2013-10-15,"[[Abu-Shammala, Wael, ], [Torchinsky, Alberto, ]]"


In [11]:
arxiv_df.iloc[0]

id                                                        0704.0001
submitter                                            Pavel Nadolsky
authors           C. Bal\'azs, E. L. Berger, P. M. Nadolsky, C.-...
title             Calculation of prompt diphoton production cros...
comments                    37 pages, 15 figures; published version
journal-ref                                Phys.Rev.D76:013009,2007
doi                                      10.1103/PhysRevD.76.013009
report-no                                          ANL-HEP-PR-07-12
categories                                                   hep-ph
license                                                        None
abstract            A fully differential calculation in perturba...
versions          [{'version': 'v1', 'created': 'Mon, 2 Apr 2007...
update_date                                              2008-11-26
authors_parsed    [[Balázs, C., ], [Berger, E. L., ], [Nadolsky,...
Name: 0, dtype: object

In [12]:
arxiv_df.isnull().sum().sort_values(ascending=False)

license           9287
report-no         9117
journal-ref       4567
doi               3637
comments          1138
id                   0
submitter            0
authors              0
title                0
categories           0
abstract             0
versions             0
update_date          0
authors_parsed       0
dtype: int64

In [14]:
(arxiv_df.isnull().sum() / len(arxiv_df)).sort_values(ascending=False)

license           0.9287
report-no         0.9117
journal-ref       0.4567
doi               0.3637
comments          0.1138
id                0.0000
submitter         0.0000
authors           0.0000
title             0.0000
categories        0.0000
abstract          0.0000
versions          0.0000
update_date       0.0000
authors_parsed    0.0000
dtype: float64

In [16]:
arxiv_df["categories"].head(20).tolist()

['hep-ph',
 'math.CO cs.CG',
 'physics.gen-ph',
 'math.CO',
 'math.CA math.FA',
 'cond-mat.mes-hall',
 'gr-qc',
 'cond-mat.mtrl-sci',
 'astro-ph',
 'math.CO',
 'math.NT math.AG',
 'math.NT',
 'math.NT',
 'math.CA math.AT',
 'hep-th',
 'hep-ph',
 'astro-ph',
 'hep-th',
 'math.PR math.AG',
 'hep-ex']

In [18]:
all_category_codes = []

for cat_str in arxiv_df["categories"].dropna():
    all_category_codes.extend(str(cat_str).split())

category_freq = Counter(all_category_codes)
category_freq.most_common(25)

[('astro-ph', 2064),
 ('hep-th', 996),
 ('hep-ph', 901),
 ('quant-ph', 672),
 ('gr-qc', 558),
 ('cond-mat.stat-mech', 445),
 ('cond-mat.mtrl-sci', 444),
 ('math-ph', 436),
 ('math.MP', 436),
 ('cond-mat.str-el', 419),
 ('cond-mat.mes-hall', 391),
 ('math.AG', 313),
 ('nucl-th', 277),
 ('cond-mat.other', 269),
 ('cond-mat.supr-con', 249),
 ('cond-mat.soft', 235),
 ('math.PR', 228),
 ('hep-ex', 224),
 ('math.DG', 209),
 ('math.CO', 205),
 ('math.AP', 190),
 ('cs.IT', 154),
 ('math.IT', 154),
 ('physics.gen-ph', 149),
 ('math.NT', 146)]

In [19]:
arxiv_df[arxiv_df["id"].isnull() | arxiv_df["title"].isnull() | arxiv_df["categories"].isnull()][["id", "title", "categories"]].head(20)

,id,title,categories


In [20]:
dot_vs_no_dot = {
    "contains_dot": sum("." in cat for cat in all_category_codes),
    "no_dot": sum("." not in cat for cat in all_category_codes),
}
dot_vs_no_dot

{'contains_dot': 8695, 'no_dot': 6369}

In [21]:
for col in ["title", "authors", "abstract", "categories"]:
    print(f"\nCOLUMN: {col}")
    for val in arxiv_df[col].head(5):
        print(repr(val))


COLUMN: title
'Calculation of prompt diphoton production cross sections at Tevatron and\n  LHC energies'
'Sparsity-certifying Graph Decompositions'
'The evolution of the Earth-Moon system based on the dark matter field\n  fluid model'
'A determinant of Stirling cycle numbers counts unlabeled acyclic\n  single-source automata'
'From dyadic $\\Lambda_{\\alpha}$ to $\\Lambda_{\\alpha}$'

COLUMN: authors
"C. Bal\\'azs, E. L. Berger, P. M. Nadolsky, C.-P. Yuan"
'Ileana Streinu and Louis Theran'
'Hongjun Pan'
'David Callan'
'Wael Abu-Shammala and Alberto Torchinsky'

COLUMN: abstract
'  A fully differential calculation in perturbative quantum chromodynamics is\npresented for the production of massive photon pairs at hadron colliders. All\nnext-to-leading order perturbative contributions from quark-antiquark,\ngluon-(anti)quark, and gluon-gluon subprocesses are included, as well as\nall-orders resummation of initial-state gluon radiation valid at\nnext-to-next-to-leading logarithmic accuracy

In [22]:
for col in ["doi", "journal-ref", "comments", "title", "authors", "abstract", "categories"]:
    empty_count = (arxiv_df[col] == "").sum()
    print(col, empty_count)

doi 0
journal-ref 0
comments 0
title 0
authors 0
abstract 0
categories 0


# Observations
1) Categories are space separated

2) some are dotted hierarchical style codes(eg: "math.CO")

3) some are just flat codes(eg: "hep-ph")

4) A paper can belong to multiple categories (most papers are multi category papers)

5) 14 columns in total and 5 of those ("license", "report-no", "journal-ref", "doi", "comments" contain null values. All other columns are fully populated)

6) Will have to drop 4 of the 5 null value containing categories above since they're sparse and won't be of much use. 

"license" and "report-no" not of much use to the functinoality anyways.
"journal-ref" and "comments" useful when present but not reliable enough to acquire :( 
"doi" useful and will preserve as optional metadata, will standardize missing values

7) Will also drop "submitter", "versions", "authors_parsed" since not too useful for current user flow


# Cleaning work

1) Normalize text fields
- remove leading and trailing whitespaace
- replacenewline characters (\n) with spaces
- replace repeated whitespace with a single whitespace
- apply this normalization to title and abstract

2) Standardize category field
- clean and trim category strings
- split multiple categories using space separation
- ensure all the categories are preserved (for papers that fall udner multiple categories)
- support both heirarchical ("math.CO") and flat ("hep-ph) formats
- parse categories into structured components to make downstream processing for tree easier

3) Data validity
- skipped malform JSON entries during streaming
- no records missing critical fields (id, title, authors, abstract, categories) so not skipping any records due to this reason

## Preprocessing validation
to see if changes work as intended before scaling script to full dataset

In [24]:
def normalize_text(text):
    if text is None:
        return None
    text = str(text).strip()
    text = re.sub(r"\s+", " ", text)
    return text if text != "" else None

In [25]:
arxiv_df["title_clean"] = arxiv_df["title"].apply(normalize_text)
arxiv_df["abstract_clean"] = arxiv_df["abstract"].apply(normalize_text)
arxiv_df["categories_clean"] = arxiv_df["categories"].apply(normalize_text)

In [26]:
arxiv_df[["title", "title_clean"]].head(5)

,title,title_clean
0,Calculation of prompt diphoton production cros...,Calculation of prompt diphoton production cros...
1,Sparsity-certifying Graph Decompositions,Sparsity-certifying Graph Decompositions
2,The evolution of the Earth-Moon system based o...,The evolution of the Earth-Moon system based o...
3,A determinant of Stirling cycle numbers counts...,A determinant of Stirling cycle numbers counts...
4,From dyadic $\Lambda_{\alpha}$ to $\Lambda_{\a...,From dyadic $\Lambda_{\alpha}$ to $\Lambda_{\a...


In [27]:
arxiv_df[["abstract", "abstract_clean"]].head(3)

,abstract,abstract_clean
0,A fully differential calculation in perturba...,A fully differential calculation in perturbati...
1,"We describe a new algorithm, the $(k,\ell)$-...","We describe a new algorithm, the $(k,\ell)$-pe..."
2,The evolution of Earth-Moon system is descri...,The evolution of Earth-Moon system is describe...


In [33]:
for i in range(3):
    print("BEFORE:")
    print(repr(arxiv_df.loc[i, "abstract"]))
    print("\nAFTER:")
    print(repr(arxiv_df.loc[i, "abstract_clean"]))
    print("\n" + "-"*80 + "\n")

BEFORE:
'  A fully differential calculation in perturbative quantum chromodynamics is\npresented for the production of massive photon pairs at hadron colliders. All\nnext-to-leading order perturbative contributions from quark-antiquark,\ngluon-(anti)quark, and gluon-gluon subprocesses are included, as well as\nall-orders resummation of initial-state gluon radiation valid at\nnext-to-next-to-leading logarithmic accuracy. The region of phase space is\nspecified in which the calculation is most reliable. Good agreement is\ndemonstrated with data from the Fermilab Tevatron, and predictions are made for\nmore detailed tests with CDF and DO data. Predictions are shown for\ndistributions of diphoton pairs produced at the energy of the Large Hadron\nCollider (LHC). Distributions of the diphoton pairs from the decay of a Higgs\nboson are contrasted with those produced from QCD processes at the LHC, showing\nthat enhanced sensitivity to the signal can be obtained with judicious\nselection of eve

In [28]:
def parse_categories(cat_str):
    if cat_str is None:
        return []

    parsed = []
    for c in cat_str.split():
        if "." in c:
            top, sub = c.split(".", 1)
        else:
            top, sub = c, None

        parsed.append({
            "full_code": c,
            "top_level": top,
            "subtopic": sub
        })
    return parsed

In [29]:
arxiv_df["categories_parsed"] = arxiv_df["categories_clean"].apply(parse_categories)

arxiv_df[["categories", "categories_parsed"]].head(5)

,categories,categories_parsed
0,hep-ph,"[{'full_code': 'hep-ph', 'top_level': 'hep-ph'..."
1,math.CO cs.CG,"[{'full_code': 'math.CO', 'top_level': 'math',..."
2,physics.gen-ph,"[{'full_code': 'physics.gen-ph', 'top_level': ..."
3,math.CO,"[{'full_code': 'math.CO', 'top_level': 'math',..."
4,math.CA math.FA,"[{'full_code': 'math.CA', 'top_level': 'math',..."


In [35]:
for i in range(5):
    print("RAW:", arxiv_df.loc[i, "categories"])
    print("PARSED:", arxiv_df.loc[i, "categories_parsed"])
    print("-"*50)

RAW: hep-ph
PARSED: [{'full_code': 'hep-ph', 'top_level': 'hep-ph', 'subtopic': None}]
--------------------------------------------------
RAW: math.CO cs.CG
PARSED: [{'full_code': 'math.CO', 'top_level': 'math', 'subtopic': 'CO'}, {'full_code': 'cs.CG', 'top_level': 'cs', 'subtopic': 'CG'}]
--------------------------------------------------
RAW: physics.gen-ph
PARSED: [{'full_code': 'physics.gen-ph', 'top_level': 'physics', 'subtopic': 'gen-ph'}]
--------------------------------------------------
RAW: math.CO
PARSED: [{'full_code': 'math.CO', 'top_level': 'math', 'subtopic': 'CO'}]
--------------------------------------------------
RAW: math.CA math.FA
PARSED: [{'full_code': 'math.CA', 'top_level': 'math', 'subtopic': 'CA'}, {'full_code': 'math.FA', 'top_level': 'math', 'subtopic': 'FA'}]
--------------------------------------------------
